In [1]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image
import kagglehub

I0000 00:00:1782410769.677694   16543 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782410769.855452   16543 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782410772.179893   16543 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
path = kagglehub.dataset_download("deepcontractor/is-that-santa-image-classification")
path = os.path.join(path, "is that santa")
print("Path to dataset files:", path)


Path to dataset files: /home/inzenfenix/.cache/kagglehub/datasets/deepcontractor/is-that-santa-image-classification/versions/5/is that santa


In [5]:
TRAIN_DIR = os.path.join(path, 'train')
VAL_DIR = os.path.join(path, 'test')

print('Directorio de trabajo:', os.getcwd())
print('TRAIN_DIR existe:', os.path.exists(TRAIN_DIR))
print('VAL_DIR existe:', os.path.exists(VAL_DIR))

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, 
    target_size=(150, 150), # Changed from (32, 32)
    batch_size=32, 
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR, 
    target_size=(150, 150), # Changed from (224, 224)
    batch_size=32, 
    class_mode='binary'
)
print("Mapeo de clases:", train_generator.class_indices)

Directorio de trabajo: /home/inzenfenix/Documents/github/framework-ia-UDD/Detector_santa
TRAIN_DIR existe: True
VAL_DIR existe: True
Found 614 images belonging to 2 classes.


Found 616 images belonging to 2 classes.
Mapeo de clases: {'not-a-santa': 0, 'santa': 1}


In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, epochs=20, validation_data=val_generator)
model.save('detector_santa_model.h5')

/home/inzenfenix/Documents/github/framework-ia-UDD/.venv_tensor/lib64/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.5668 - loss: 1.4579 - val_accuracy: 0.6786 - val_loss: 0.9113
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 942ms/step - accuracy: 0.7948 - loss: 0.7642 - val_accuracy: 0.8458 - val_loss: 0.5961
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 20s 982ms/step - accuracy: 0.8599 - loss: 0.6085 - val_accuracy: 0.8750 - val_loss: 0.5074
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 974ms/step - accuracy: 0.8664 - loss: 0.5230 - val_accuracy: 0.8734 - val_loss: 0.5035
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 940ms/step - accuracy: 0.8779 - loss: 0.4525 - val_accuracy: 0.8880 - val_loss: 0.3897
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 904ms/step - accuracy: 0.8844 - loss: 0.4242 - val_accuracy: 0.9075 - val_loss: 0.3426
Epoch 7/20
14/20 ━━━━━━━━━━━━━━━━━━━━ 3s 537ms/step - accuracy: 0.8697 - loss: 0.3896

In [ ]:
def clasificar_imagen(ruta_imagen):
    img = image.load_img(ruta_imagen, target_size=(150, 150))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    probabilidad = model.predict(img_array)[0][0]
    if probabilidad > 0.5:
        print(f"¡Es SANTA Claus! 🎅 ({probabilidad*100:.2f}%)")
    else:
        print(f"No es Santa ❌ ({(1 - probabilidad)*100:.2f}%)")

# Para probar una foto, solo cambia el nombre aquí y corre esta celda:
clasificar_imagen('./dog_santa.jpg')